## PROCESSAMENTO DOS DADOS
Este notebook visa ler os dados de bares e restaurantes, extrair suas coordenadas geográficas usando a API da ```geocode.maps.co``` e formar um novo conjunto de dados apenas com as informações necessárias.

### 1- Leitura e Filtragem inicial
Aqui usamos a biblioteca pandas para ler o arquivo ```bares.csv``` e filtrar algumas colunas que não vamos precisar, além de mostrar quais os tipos de descrição de logradouro, pois é necessário convertê-los para o nome completo na hora da query.

In [2]:
import pandas as pd
from pathlib import Path

In [3]:
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
PRC_DIR = ROOT / "data" / "processed"

In [4]:
bar_table = pd.read_csv(f"{RAW_DIR}/bares.csv")
colunas_rem = ["CNAE_PRINCIPAL", "DESCRICAO_CNAE_PRINCIPAL"]
bar_table = bar_table.drop(columns=colunas_rem)
bar_table.head(5)

,_id,DESC_LOGRADOURO,NOME_LOGRADOURO,NUMERO_IMOVEL,COMPLEMENTO,NOME_BAIRRO,NOME,NOME_FANTASIA
0,12,AVE,PORTUGAL,4316,NaN,JARDIM ATLANTICO,BAR E CAFE DONATELO LTDA,NaN
1,32,RUA,VINTE E OITO DE SETEMBRO,606,NaN,ESPLANADA,PALLATE COMERCIO DE PRODUTOS ALIMENTICIOS LTDA,PALLATE
2,40,AVE,EDGAR DA MATA MACHADO,390,A,DOM CABRAL,FABIO KALKS KOSCKY,BAR DO FABIO
3,58,AVE,DOS BANDEIRANTES,1839,NaN,CRUZEIRO,NECTAR DA SERRA LTDA,NECTAR DA SERRA
4,60,RUA,DO OURO,1428,NaN,SERRA,MERCEARIA DA SERRA LTDA,BAR E MERCEARIA DA SERRA


In [5]:
bar_table[["_id", "DESC_LOGRADOURO"]].groupby("DESC_LOGRADOURO").count()

,_id
DESC_LOGRADOURO,
ALA,32
AVE,4604
BEC,114
EST,25
PCA,190
ROD,171
RUA,15709
TRV,10
VDP,1


In [6]:
null_report = (bar_table.isna().mean().mul(100).round(2).sort_values(ascending=False).reset_index())
null_report.columns = ["column", "null_percent"]
null_report[null_report["null_percent"] > 0].head(20)

,column,null_percent
0,COMPLEMENTO,52.98
1,NOME_FANTASIA,45.29


### 2- Extração das Coordenadas
A extração das coordenadas a partir dos endereços é feita pela API da ```geocode.maps.co```. É necessário ter uma conta e ajustar os valores dentro das células de acordo com a sua conta.
No entanto, não recomendo rodar o script se ```cords.json``` já existir em ```data/processed```, pois leva horas para extrair cada coordenada geográfica de todos os endereços.

In [7]:
import time
import json
import os
import requests as req

In [ ]:
CORDS_DIR = f"{PRC_DIR}/cords.json"
FDATA_DIR = f"{PRC_DIR}/fulldata.json"

api_key = ""     # Coloque a sua API key aqui
sleep_time = 0.23       # 0.22-0.25 se estiver dentro do limite de 25 mil pesquisas da conta grátis, 1 caso contrário.
CONSULTAS = len(bar_table)   # len(bar_table) para tudo

url = "https://geocode.maps.co/search"
cols = ["DESC_LOGRADOURO", "NOME_LOGRADOURO", "NUMERO_IMOVEL"]

# Dicionários que serão escritos nos json mais tarde
if os.path.exists(CORDS_DIR):
    try:
        with open(CORDS_DIR, "r", encoding="utf-8") as f:
            cords_data = json.load(f)
    except Exception as e:
        print("Unable to load previous data:", e)
        cords_data = {}

else:
    cords_data = {}

# cords_full_data = {}

# Mantém contagem de sucessos e falhas nas consultas
successes = 0
fails = 0

# Consultas
for i in range(CONSULTAS):
    # Caso o endereço já tenha sido processado
    if str(i) in cords_data:
        continue

    bar = bar_table.iloc[i]
    bar_name = bar["NOME"]
    bar_fantasia = bar["NOME_FANTASIA"]
    bar_fantasia = str(bar_fantasia) if pd.notna(bar_fantasia) else ""

    desc, logr, numi = [bar[cols[j]] for j in range(len(cols))]

    # Conversão do nome inteiro do tipo de logradouro
    match desc:
        case "AVE":
            desc = "AVENIDA"
        case "ALA":
            desc = "ALAMEDA"
        case "BEC":
            desc = "BECO"
        case "EST":
            desc = "ESTRADA"
        case "PCA":
            desc = "PRAÇA"
        case "ROD":
            desc = "RODOVIA"
            logr = "BR356"
        case "RUA":
            desc = "RUA"
        case "TRV":
            desc = "TRAVESSA"
        case "VDP":
            desc = ""
        case "VIA":
            desc = "VIADUTO"

    address = f"{desc} {logr}, {numi}, Belo Horizonte, Minas Gerais, Brazil"
    params = {"q": address, "api_key": api_key}

    success = False

    for t in range(2):
        try:
            query = req.get(url, params=params, timeout=10)
            query.raise_for_status()
            data = query.json()

            # cords_full_data[str(i)] = {"NOME": bar_name, "NOME_FANTASIA": bar_fantasia, "ENDERECO": address, "GEOCODE": data}
            if data:
                lat = data[0].get("lat")
                lon = data[0].get("lon")
                cords_data[str(i)] = {"NOME": bar_name, "NOME_FANTASIA": bar_fantasia, "ENDERECO": address, "LATITUDE": lat, "LONGITUDE": lon}

            if i%200 == 0:  # Evita escrita em disco todo loop
                with open(CORDS_DIR, "w", encoding="utf-8") as f:
                    json.dump(cords_data, f, ensure_ascii=False, indent=4)

                # with open(FDATA_DIR, "w", encoding="utf-8") as f:
                    # json.dump(cords_full_data, f, ensure_ascii=False, indent=4)

            success = True
            successes += 1
            print(f"[{i}] {bar_name}: query successful")

            break

        except Exception as e:
            print(f"Failed line {i} - {bar_name}: {e}")
            time.sleep(sleep_time)

    # Falha crítica - nenhuma das tentativas sucedeu
    if success == False:
        print(f"Critical failure on line {i}")
        fails += 1

    time.sleep(sleep_time)    # Limite de consultas da API

# Dump final
with open(CORDS_DIR, "w", encoding="utf-8") as f:
    json.dump(cords_data, f, ensure_ascii=False, indent=4)

# with open(FDATA_DIR, "w", encoding="utf-8") as f:
    # json.dump(cords_full_data, f, ensure_ascii=False, indent=4)

print("Sucessos:", successes)
print("Falhas:", fails)